<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex10.2-solid-oxide-cell/Ex10.2_03_degradation_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

### A correction to how this is framed

An earlier version of this notebook asked for hydrogen production against
lifetime as the temperature varies, at fixed current density. **That question
has a trivial answer**, and it is worth seeing why before going further.

Hydrogen production follows Faraday's law,

$$
\dot n_{\mathrm{H_2}} = \frac{I\,A}{2F}
$$

so at fixed current density the production rate is **fixed**, whatever the
temperature does. Raising the temperature lowers the resistance and therefore
the voltage, so it reduces the *energy consumed per unit of hydrogen*. It does
not produce more hydrogen.

The questions worth asking are therefore:

- **energy consumption against degradation** at fixed current, where temperature
  genuinely trades one against the other; or
- a **current density sweep**, in which production does change, and degradation
  changes with it.

A second point of the same kind. The degradation law goes as $|i|^n$, so
sweeping the exponent at $i = 1$ A/cm² changes nothing at all: $1^n = 1$ for
every $n$. Sweep the exponent at a current density away from unity, or the
result will be identically flat.


# Ex_10.2 · Notebook 03 — degradation and the trade-off

**Paired with L10.2 · Solid oxide cells**

Produce the trade-off curve as a **result** rather than accepting it from a
slide: production rate against expected lifetime, as temperature varies.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex10.2-solid-oxide-cell/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## TODO — sweep temperature and plot both quantities

In [ ]:
ASR_LIMIT = 0.50          # end-of-life threshold, ohm cm2
I_OP = 1.0                # A/cm2

rows = []
for T_c in (650, 700, 750, 800, 850):
    par = pb.SOCParams("SOEC", T_c + 273.15, 0.10, 0.90)
    rate = pb.degradation_rate(I_OP, par)
    hours_to_eol = (ASR_LIMIT - par.ASR_0) / max(rate, 1e-12)
    V = pb.cell_voltage(I_OP, par)
    h2 = pb.hydrogen_rate(I_OP, par) * 3600
    rows.append((T_c, V, h2, rate, hours_to_eol))
    print(f"  {T_c} C:  V {V:.3f} V   H2 {h2:.3e} mol/h   "
          f"deg {rate:.2e}   life {hours_to_eol:,.0f} h")

# TODO 1 --- production against lifetime ---------------------------------------------------------------------
# Two `...` to replace:
#   line 1  ->  [h * L for _, _, h, _, L in rows]        total hydrogen over life = rate x lifetime
#   line 2  ->  T_c_list[int(np.argmax(total))]          the temperature with the largest total
T_c_list = [r[0] for r in rows]
h2_list  = [r[2] for r in rows]
life     = [r[4] for r in rows]
total    = ...                                    # <- [h * L for _, _, h, _, L in rows]
T_best   = ...                                    # <- T_c_list[int(np.argmax(total))]

fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.plot(T_c_list, h2_list, "o-", color=CYCLE[0], label="H2 rate [mol/h]")
ax.set_xlabel("temperature [degC]"); ax.set_ylabel("H2 rate [mol/h]", color=CYCLE[0])
ax2 = ax.twinx()
ax2.semilogy(T_c_list, life, "s--", color=CYCLE[1], label="life to ASR limit [h]")
ax2.set_ylabel("life [h]", color=CYCLE[1])
ax.axvline(T_best, color="#888888", ls=":", label=f"max total H2 at {T_best} C")
ax.set_title("Faster now, or for longer"); ax.legend(frameon=False, fontsize=9, loc="upper left")
plt.tight_layout(); plt.show()

print(error_table([[f"{T:.0f}", f"{h:.3e}", f"{L:,.0f}", f"{h*L:.3e}"]
                   for T, _, h, _, L in rows],
                  ["T [degC]", "H2 [mol/h]", "life [h]", "total [mol]"]))
# ------------------------------------------------------------------------------

**The question this notebook exists to ask.** Total hydrogen produced over the
device's life is production rate multiplied by lifetime. Neither the hottest
nor the coolest setting maximises it. Where is the optimum, and how sensitive
is it to the ESTIMATED degradation exponent?

Change `deg_exponent` from 1.5 to 1.0 and 2.0 and repeat. If the optimum moves
a lot, say so in your report — that is a result about your *model*, not about
solid oxide cells.